# SUPPORT2 Dataset: Data Cleaning & Preparation
**AAI-500 Applied Statistics for AI | Final Team Project**

---

## Notebook Objectives

This notebook covers the **Data Cleaning & Preparation** phase of the project pipeline:

1. Initial data inspection and profiling
1. Column renames per PEP-8
1. Engineering the binary target variable (`death_180d`)
1. Data type casting and ordinal/nominal encoding
1. Missing data analysis and imputation
1. Dropping unused columns
1. Documenting target / feature / outcome 
1. Export of the cleaned dataset for EDA and modeling

---
## Section 0: Setup & Imports

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


sys.path.append("../../utils")
from dataset import get_project_root, load_csv

warnings.filterwarnings("ignore")

current = Path.cwd().resolve()

while current != current.parent:
    if (current / ".git").exists() or (current / "pyproject.toml").exists():
        project_root = current
        break
    current = current.parent

PROJ_ROOT = get_project_root()
OUTPUT_PATH = PROJ_ROOT / "data" / "support2_cleaned.csv"
print(f"Output path  : {OUTPUT_PATH}")

---
## Section 1: Load Data & Initial Inspection

In [ ]:
df_raw = load_csv("support2_raw_complete.csv")
print(f"Shape: {df_raw.shape[0]:,} rows  x  {df_raw.shape[1]} columns")
df_raw.head()

In [ ]:
df_raw.info(verbose=True, show_counts=True)

In [ ]:
int_cols = df_raw.select_dtypes(include="int64").columns
binary_cols = [
    c
    for c in int_cols
    if df_raw[c].nunique() == 2 and set(df_raw[c].unique()) == {0, 1}
]
print("===========Binary Columns==========")
print(binary_cols)


### 1.1 Descriptive Statistics

In [ ]:
desc = df_raw.describe().T
desc.columns = [
    "Count",
    "Mean",
    "Std Dev",
    "Min",
    "Q1 (25%)",
    "Median (50%)",
    "Q3 (75%)",
    "Max",
]
desc

---
## Section 2: Column Renaming

Rename some columsn to follow PEP-8 standard

In [ ]:
rename_map = {
    "d.time": "d_time",
    "num.co": "num_co",
}
df = df_raw.rename(columns=rename_map).copy()

print("Column renames applied:")
for old, new in rename_map.items():
    print(f"  {old!r}  ->  {new!r}")
print(f"\nTotal columns: {len(df.columns)}")

---
## Section 3: Target Variable

**Did the patient die within 180 days of study?**

To answer this we look at two fields: 
- `death`:  whether the patient ever died during 180 days (1 = yes, 0 = no)
- `d_time`:  days of follow-up (how long they were observed). If this number is greater than 180 days of the trial, that means they survived the trial

death_180d (1 = yes, 0 = no) = {death=1 AND d_time <= 180}

Notice the AND conditional because the patient must've died within the 180 days to satisfy our target. 
We then use **death_180d** for binary classification


In [ ]:
df["death_180d"] = ((df["death"] == 1) & (df["d_time"] <= 180)).astype(int)

counts = df["death_180d"].value_counts()
rel_freqs = df["death_180d"].value_counts(normalize=True)

summary = pd.DataFrame(
    {
        "Outcome": ["Survived >= 180 days (0)", "Died within 180 days (1)"],
        "Frequency": counts.values,
        "Relative Frequency": rel_freqs,
        "Percentage (%)": (rel_freqs.values * 100),
    }
)
print(summary.to_string(index=False))
print(f"\nClass imbalance ratio (survived: died) = {counts[0] / counts[1]:.2f} : 1")

---
## Section 4: Data Type Casting (into Pandas Type)

Correct data types are essential for proper computing and modeling.
We apply three categories of correction. Casting to Int8 also handles NaN gracefully:

1. **Binary integer columns** -> `Int8` (nullable integer; memory-efficient)
2. **Nominal categorical columns** -> unordered `pd.Categorical`
3. **Ordinal categorical columns** -> ordered `pd.Categorical` for proper sorting and comparision

**`income` ordering (ascending socioeconomic status):**
`under $11k` < `$11-$25k` < `$25-$50k` < `>$50k`

**`sfdm2` ordering (best functional status → worst):**
`no disability` < `moderate ADL impairment` < `severe SIP impairment` < `coma/intubated` < `died before 2-month interview`

In [ ]:
# Binary columns -> nullable Int8
binary_cols = ["death", "hospdead", "diabetes", "dementia"]
for col in binary_cols:
    df[col] = df[col].astype("Int8")

######  8 Catagoricals
# sex = 2 levels
# dzgroup = 5
# dzclass = 4
# race = 5
# income = 4
# ca = 3
# dnr = 3
# sfdm2 = 5
##############

# Nominal (unordered) categoricals
nominal_cats = ["sex", "dzgroup", "dzclass", "race", "ca", "dnr"]
for col in nominal_cats:
    df[col] = df[col].astype("category")

# Ordinal: income (lowest -> highest bracket) as coded in support2_data_dictionary.md
income_order = ["under $11k", "$11-$25k", "$25-$50k", ">$50k"]
df["income"] = pd.Categorical(df["income"], categories=income_order, ordered=True)

# Ordinal: sfdm2 (2-month functional disability; no disability -> died before interview)
# as coded in support2_data_dictionary.md
sfdm2_order = [
    "no(M2 and SIP pres)",  # Level 1 - no moderate/severe disability
    "adl>=4 (>=5 if sur)",  # Level 2 - unable to do 4+ ADL
    "SIP>=30",  # Level 3 - Sickness Impact Profile >= 30
    "Coma or Intub",  # Level 4 - intubated or in coma at 2 months
    "<2 mo. follow-up",  # Level 5 - died before 2-month interview
]
df["sfdm2"] = pd.Categorical(df["sfdm2"], categories=sfdm2_order, ordered=True)

print("Data type corrections applied:\n")
cols_updated = binary_cols + nominal_cats + ["income", "sfdm2"]
print(df[cols_updated].dtypes.to_string())

---
## Section 5: Missing Data Analysis

The authors have stated there are a maximum of 5641 N/As. This, along with every other missing data, we must verify and address.


In [ ]:
missing_count = df.isnull().sum()
missing_percent = missing_count / len(df) * 100
missing_df = pd.DataFrame(
    {
        "Missing Count": missing_count,
        "Missing %": missing_percent,
    }
).sort_values("Missing Count", ascending=False)

missing_df = missing_df[missing_df["Missing Count"] > 0]

total_cells = df.shape[0] * df.shape[1]
total_missing = int(missing_df["Missing Count"].sum())

print(f"Columns with at least one missing value : {len(missing_df)}")
print(
    f"Total missing cells : {total_missing:,} / {total_cells:,}  ({total_missing / total_cells * 100:.1f}%)\n"
)
print(missing_df.to_string())

# verify maximum missing n/as
max_na = df_raw.isnull().sum().max()
max_na_col = df_raw.isnull().sum().idxmax()
print(f"Max NAs: {max_na} (column: '{max_na_col}')")
assert max_na == 5641, f"Expected 5641, got {max_na}"

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.barh(
    missing_df.index[::-1],
    missing_df["Missing %"][::-1],
    edgecolor="white",
    height=0.7,
)


ax.set_xlabel("Missing Data (%)", fontsize=11)
ax.set_title("Missing Data Rate by Column", fontsize=13, fontweight="bold")
ax.set_xlim(0, 80)

for bar, pct in zip(bars, missing_df["Missing %"][::-1]):
    if pct > 0:
        ax.text(
            bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%",
            va="center",
            fontsize=8,
        )
plt.show()

### 5.1 Missing Data Decision Table

Here we decide how to tackle missing data. Either drop the column entirely or impute mathematically

For most of our biological features that have missing data, we compute the median. The reason is per Agresti et al (2022), chapter 3, that for right-skewed data, using median instead of mean worse the skewness of the data (push it woard the tail). Looking at section 1, our columns the mean > median, which signals right skewed.



| Column(s) | Decision | Rationale |
|---|---|---|
| `adlp`, `adls` | **Drop** | Superseded by `adlsc` per original authors
| `dnrday` | **Drop** | Do-no-resus day. Useless for our purpose
| `hday` | **Drop** | Day admitted to study. Nothing to do with mortality
| `charges, totcst, totmcst` | **Drop** | Relating to hospital cost, which we won't be studying
| `alb`, `pafi`, `bili`, `crea`, `bun`, `wblc`, `urine` | **imputed values avail** | Per Author's note we have the recommended imputed values
| `ph`, `glucose`, `scoma`, `avtisst`, `edu`, `prg2m`, `prg6m`  | **Sample median** | high missing |
| `surv2m`, `surv6m`, `aps`, | **Sample median** | kept as benchmark columns |
| `meanbp`, `hrt`, `resp`, `temp`, `sod` | **Sample median** | few missing |
| `charges`, `totcst` |  **Sample median** | Retained for future cost analysis |
| `totmcst` | **Sample median** | Retained for future cost analysis; many missing values and will be difficult to impute. Contains negative values (balances) |
| `race`, `income`, `sfdm2`, `dnr` | **Mode imputation** | Most frequent category as best single-value estimate for categoricals |

#### Benchmark Columns (kept but will be excluded from death_180d model features)

| Column | Role | Notes |
|---|---|---|
| `surv2m`, `surv6m` | SUPPORT model predictions | will be compared against our model |
| `aps` | APACHE III severity score | Independent benchmark system widely used in critical care |
| `sps` | SUPPORT physiology sub-score | Component of the SUPPORT model ?? |
| `prg2m`, `prg6m` | Physician survival estimates | ML vs. clinician judgment comparison |
| `dnr` | DNR order status | **Should not use as predictor feature as it's reflects human behavior rather than biological factors** |

---
## Section 6: Feature Pruning

We remove features not used in our analysis:

In [ ]:
cols_to_drop = ["adlp", "adls", "dnrday", "hday", "charges", "totcst", "totmcst"]

df.drop(columns=cols_to_drop, inplace=True, errors="ignore")
print(f"Dropped  : {len(cols_to_drop)} columns")
print(f"Remaining: {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print(f"Remaining columns ({df.shape[1]}):")
print(list(df.columns))

---
## Section 7: Missing Data Imputation

The original SUPPORT study authors (Harrell, 2022) provide clinically validated
**normal fill-in values** for seven physiological variables (see docs/support2_dataset_description.md).
We will use this for imputation

| Variable | Normal Fill | Meaning |
|---|---|---|
| `alb`   | 3.5   | Serum albumin - liver / nutrition |
| `pafi`  | 333.3 | PaO2/FiO2 ratio - oxygen level|
| `bili`  | 1.01  | Bilirubin - liver function |
| `crea`  | 1.01  | Creatinine - renal function |
| `bun`   | 6.51  | BUN - normal renal function |
| `wblc`  | 9.0   | WBC count - immune status |
| `urine` | 2502  | Urine output - renal output |

In [ ]:
DOMAIN_FILL = {
    "alb": 3.5,
    "pafi": 333.3,
    "bili": 1.01,
    "crea": 1.01,
    "bun": 6.51,
    "wblc": 9.0,
    "urine": 2502.0,
}

imputation_log = []

for k, v in DOMAIN_FILL.items():
    n_missing = int(df[k].isnull().sum())
    if n_missing > 0:
        df[k] = df[k].fillna(v)
        imputation_log.append(
            {
                "Column": k,
                "Strategy": "Domain Fill",
                "Fill Value": v,
                "Cells Filled": n_missing,
            }
        )

MEDIAN_COLS = [
    "meanbp",
    "hrt",
    "resp",
    "temp",
    "sod",
    "ph",
    "glucose",
    "scoma",
    "avtisst",
    "edu",
    "surv2m",
    "surv6m",
    "aps",
    "prg2m",
    "prg6m",
]
for k in MEDIAN_COLS:
    n_missing = int(df[k].isnull().sum())
    if n_missing > 0:
        med = float(df[k].median())
        df[k] = df[k].fillna(med)
        imputation_log.append(
            {
                "Column": k,
                "Strategy": "Sample Median",
                "Fill Value": round(med, 4),
                "Cells Filled": n_missing,
            }
        )

MODE_COLS = ["race", "income", "sfdm2", "dnr"]

for k in MODE_COLS:
    n_missing = int(df[k].isnull().sum())
    if n_missing > 0:
        mode_val = df[k].mode()[0]
        df[k] = df[k].fillna(mode_val)
        imputation_log.append(
            {
                "Column": k,
                "Strategy": "Sample Mode",
                "Fill Value": str(mode_val),
                "Cells Filled": n_missing,
            }
        )

log_df = pd.DataFrame(imputation_log)
print("Imputation Summary:")
print(log_df.to_string(index=False))
print(f"Total cells imputed: {log_df.get('Cells Filled', np.array([])).sum():,}")

In [ ]:
remaining_na = df.isnull().sum()
remaining_na = remaining_na[remaining_na > 0]

if len(remaining_na) == 0:
    print("Imputation complete, zero missing values remain in any column.")
else:
    print("WARNING: Remaining missing values:")
    print(remaining_na.to_string())

print(
    f"\nDataset shape after imputation: {df.shape[0]:,} rows  x  {df.shape[1]} columns"
)

---
## Section 8: Defining Feature, Outcome Separation, Leakage 

| Category | Columns |
|---|---|
| **Target** | `death_180d` |
| **Outcome columns** (excluded due to result leakage) | `death`, `hospdead`, `d_time`, `slos` |
| **Row identifier** | `id` | 
| **Features** | All remaining columns | 

---
## Section 9: Post-Cleaning Summary

We compare the dataset before and after cleaning and present the five-number
summary for key physiological lab variables to confirm that imputation and
outlier retention produced a coherent, analysis-ready dataset.

In [ ]:
print("=== Pre-Cleaning ===")
print(f"  Shape          : {df_raw.shape[0]:,} rows  x  {df_raw.shape[1]} columns")
print(f"  Missing cells  : {df_raw.isnull().sum().sum():,}")

print("\n=== Post-Cleaning ===")
print(f"  Shape          : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print(f"  Missing cells  : {df.isnull().sum().sum()}")
print(f"  Rows retained  : {df.shape[0] / df_raw.shape[0] * 100:.1f}%")

prev = df["death_180d"].mean()

### 9.1 Important Lab Variables Summary (Post-Cleaning)

In [ ]:
key_labs = [
    "age",
    "meanbp",
    "hrt",
    "temp",
    "alb",
    "bili",
    "crea",
    "glucose",
    "bun",
    "urine",
]
post_desc = df[key_labs].describe().T
post_desc.columns = ["Count", "Mean", "Std Dev", "Min", "Q1", "Median", "Q3", "Max"]
post_desc[["Min", "Q1", "Median", "Q3", "Max", "Mean", "Std Dev"]]

---
## Section 10: Save Cleaned Dataset

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to : {OUTPUT_PATH}")
print(f"Final shape              : {df.shape[0]:,} rows  x  {df.shape[1]} columns")
print("\nColumn list and dtypes:")
print(df.dtypes.to_string())

## Imputation Bias Check

Several laboratory columns carried substantial missingness before imputation
(glucose, albumin, and bilirubin roughly 25 to 49 percent), while creatinine and
white-cell count were nearly complete. This check examines whether that missingness
was related to the outcome, which is the missing-at-random assumption the
downstream models depend on. The raw table is read only to recover the original
missing positions, then mortality is compared between originally-missing and
originally-present rows.

In [ ]:
# imputation bias check (moved here from EDA per PR #9 review)
# read raw only to recover the original missing positions, then compare mortality
raw = load_csv("support2_raw_complete.csv")
dtime = "d.time" if "d.time" in raw.columns else "d_time"
y_raw = ((raw["death"] == 1) & (raw[dtime] <= 180)).astype(int)

report = []
for col in ["alb", "bili", "pafi", "glucose", "wblc", "crea"]:
    if col not in raw.columns:
        continue
    missing = raw[col].isna()  # rows where this lab was originally missing (pre-imputation)
    report.append({
        "variable": col,
        "pct_imputed": round(missing.mean() * 100, 1),
        # gap = mortality of originally-missing minus originally-present rows;
        # near 0 means missingness is unrelated to the outcome (supports the fill)
        "gap": round(y_raw[missing].mean() - y_raw[~missing].mean(), 2),
    })
print(pd.DataFrame(report).sort_values("pct_imputed", ascending=False).to_string(index=False))

The mortality gap between originally-missing and originally-present rows stayed
within 0.05 for every column examined (the largest, creatinine at 0.05, rests on
only 0.7 percent missingness and is effectively noise). The heavily-imputed columns
(glucose, albumin, bilirubin) show no meaningful link between being missing and the
outcome, so the normal-value fill does not appear to bias the outcome association.
This supports modeling on the imputed columns and provides the missing-data note for
the Limitations section.